# 第6回　CSVを読み込む・整形・記述統計
## ―― なぜ人間に読みやすい表は、機械に読ませられないのか

情報活用Ⅰ　／　北星学園大学　2026年度後期

今日やることは3つ。

1. **汚い表を読み込ませて、失敗するところを見る**
2. きれいなCSVを読み込んで、基本統計量を出す
3. **同じ調査をもう一度やったら、同じ数字が出るのか**を確かめる

> コードはAIに書かせてよい。ただし **読んで、何をしているか説明できること**。
> 説明できないコードの出力は、あなたの根拠にならない。

In [ ]:
# 準備：ライブラリと、練習用データ（北辰大学の学生400人）を読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    # ネットから取れないときは、同じデータをその場で作る（中身は気にしなくてよい）
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

print("読み込めた行数:", len(df))
df.head()

---
## 1. きれいなデータとは何か

**機械判読可能なデータ**には、4つの条件がある。

| 条件 | 意味 |
|---|---|
| **1行1件** | 1行が1人分（1件分）。1行に2人分を詰めない |
| **1列1変数** | 1列が1つの項目。「学部・学年」を1列に混ぜない |
| **セルを結合しない** | 見出しをまたぐ結合は、読み込むと空欄になる |
| **色で意味を持たせない** | 赤いセル＝要注意、は機械に伝わらない。列を作る |

さらにもう1つ。**単位を混ぜない。**「7.1時間」と「7時間半」が同じ列にあると、その列は数値ではなく文字列になり、平均が計算できなくなる。

ここまでは言葉の説明。次のセルで、実際に壊れるところを見る。

In [ ]:
# 「人間が読みやすいように」作られた表を、そのまま再現する。
# ―― 見出しの上にタイトル行、結合セルのつもりの空欄、単位つきの数字、備考行。
dirty = """2026年度 学生アンケート集計表,,,
作成: 情報活用Ⅰ,,,
,,,
学部,人数,平均睡眠時間,備考
経済学部,166,7.1時間,
,,,※一人暮らしを含む
文学部,138,7時間半,
社会福祉学部,96,,未集計
"""
with open("dirty.csv", "w", encoding="utf-8") as f:
    f.write(dirty)

# これを普通に読み込んでみる
bad = pd.read_csv("dirty.csv")
bad

見出しが `2026年度 学生アンケート集計表` になってしまった。
**1行目を見出しだと思い込む**のが `read_csv` の既定の動作だからだ。

では、見出しの位置を教えてやれば直るだろうか。

In [ ]:
# 見出しは4行目（0から数えて3行目）にある、と教える
bad2 = pd.read_csv("dirty.csv", skiprows=3)
print(bad2)
print()
print("平均睡眠時間の型:", bad2["平均睡眠時間"].dtype)
print()
# 数値として平均を出そうとすると――
try:
    print(bad2["平均睡眠時間"].mean())
except Exception as e:
    print("失敗:", type(e).__name__, e)

表の形にはなった。しかし **`平均睡眠時間` の型は `object`（＝文字列）** のままで、平均が出せない。

原因は3つとも、人間の都合でつけた飾りである。

- `7.1時間` `7時間半` … **単位が文字として混ざっている**（しかも表記がバラバラ）
- `※一人暮らしを含む` の行 … **1行1件が壊れている**（この行は誰のデータでもない）
- 社会福祉学部の空欄 … **欠測なのか0なのか、区別がつかない**

> **人が読みやすく作った表ほど、機械には読めない。**
> あなたがこれから作る集計表も、次の誰か（＝来週の自分）が読み込むことになる。

---
## 2. きれいなCSVを読み込む

最初に読み込んだ北辰大学のデータは、条件を満たしている。読み込みは **1行** で終わる。

In [ ]:
# 何行・何列あるか
print("行数（人数）:", len(df))
print("列数（項目数）:", len(df.columns))
print()
print("列の名前:")
for col in df.columns:
    print("  -", col)

### 欠測値 ―― 空欄はどこにあるか

実際の調査データには、必ず空欄が出る。答えたくない設問、飛ばされた設問、回収時の事故。
**まず「どこが空欄か」を数える。**

In [ ]:
# 列ごとに、空欄がいくつあるか
missing = df.isna().sum()
print(missing[missing > 0])
print()
print(f"400人中、どこか1つでも空欄がある人: {df.isna().any(axis=1).sum()} 人")

空欄の扱い方は2つある。**どちらを選んだかを記録しておくこと。**

- **その人を除く**（`dropna`）… 人数が減る。何人減ったかを書く
- **そのまま計算する** … pandasの `mean()` は空欄を自動で飛ばす。**分母が列ごとに違う**ことに注意

In [ ]:
# 空欄を飛ばして計算した場合、分母（有効な人数）は列ごとに違う
print("睡眠時間の平均:", round(df["睡眠時間h"].mean(), 2), " ← 有効", df["睡眠時間h"].notna().sum(), "人")
print("SNS時間の平均 :", round(df["SNS時間h"].mean(), 2), " ← 有効", df["SNS時間h"].notna().sum(), "人")

### 外れ値 ―― ありえない数字を見つける

`describe()` は、数値の列の要約を一気に出す。**最大値と最小値を必ず見る。**

In [ ]:
df[["身長cm", "睡眠時間h", "通学時間min", "テスト点"]].describe().round(1)

**身長の最大値がおかしい。** 単位を間違えて入力した人がいる（cmのつもりがmm）。

これを放置すると、平均身長が跳ね上がる。実際に確かめる。

In [ ]:
print("そのまま       :", round(df["身長cm"].mean(), 1), "cm")

# ありえない値を除く（250cmより大きい人はいない、と判断した）
clean = df[df["身長cm"] < 250]
print("外れ値を除いた :", round(clean["身長cm"].mean(), 1), "cm")
print()
print("除いた人:")
print(df[df["身長cm"] >= 250][["学生ID", "身長cm"]])

たった1人で、平均が変わった。

> **外れ値は「消す」のではなく「見つけて、判断して、記録する」。**
> 除いたなら「1名を単位ミスと判断して除外した」と報告書に書く。書いていない除外は、改ざんと区別がつかない。

---
## 3. 基本統計量を出す

報告書に必ず載せる4つ ―― **件数・平均・中央値・標準偏差**。

In [ ]:
col = "睡眠時間h"        # ← ここを変えれば他の列も見られる

print(f"【{col}】")
print(f"  件数     : {df[col].count()} 人")
print(f"  平均     : {df[col].mean():.2f}")
print(f"  中央値   : {df[col].median():.2f}")
print(f"  標準偏差 : {df[col].std():.2f}   ← 平均からどれくらい散らばっているか")
print(f"  最小/最大: {df[col].min():.1f} / {df[col].max():.1f}")

**標準偏差は「ばらつきの大きさ」。** 平均だけでは、全員が同じくらいなのか、バラバラなのかが分からない。報告書には平均と標準偏差をセットで書く。

---
## 4. 同じ調査をもう一度やったら、同じ数字が出ると思いますか

あなたが集めたデータは、**世の中の一部を切り取ったもの**でしかない。
もう一度集め直したら、同じ数字にはならない。**どれくらい違うのか**を、目で見る。

400人から **5人だけ**取り出して、平均を出す。これを5回くり返す。

In [ ]:
sleep = df["睡眠時間h"].dropna()

print("5人だけで平均を出す（5回くり返す）")
for i in range(5):
    print(f"  {i+1}回目: {sleep.sample(5).mean():.2f} 時間")

print()
print(f"（400人全員だと {sleep.mean():.2f} 時間）")

**毎回ちがう数字が出た。** 同じ集団から取り出しているのに、である。

では、取り出す人数を **30人** に増やすとどうなるか。

In [ ]:
print("30人で平均を出す（5回くり返す）")
for i in range(5):
    print(f"  {i+1}回目: {sleep.sample(30).mean():.2f} 時間")

print()
print(f"（400人全員だと {sleep.mean():.2f} 時間）")

In [ ]:
# ばらつきの大きさを、数字で比べる
for n in [5, 30, 100]:
    means = [sleep.sample(n).mean() for _ in range(300)]
    print(f"{n:>3}人で平均を出す → 300回試したときの、平均値のばらつき（標準偏差）: {np.std(means):.3f}")

**人数を増やすほど、ばらつきは小さくなる。**

これが「集めたデータは一部でしかない。数字には必ずぶれがある」ということ。統計学ではこのぶれを**誤差**と呼び、第10回で「では、どこからが『差がある』と言えるのか」を扱う。

### あなたの調査につなげる

- **第8回**：回答が3人・5人の学年を1位・2位として並べたランキングは、何を意味するのか
- **第10回**：条件をそろえた比較。「差がある」と言えるのはどんなときか
- **第13回の報告書**：あなたの回収数は、そう多くない。**「この人数では、どこまで言えるか」**を書くための材料が、いま手に入った

> 教科書『はじめて学ぶ 数理・データサイエンス・AI』第9章-03「一部のデータでは誤差を加味しよう」がそのまま対応する。

---
## 5. 卒業課題 ―― 3通りの方法で、同じ数字が出るか

同じ集計を、次の3通りで出して**一致するか**を確かめる。

1. **Google Forms の自動集計**（フォームの「回答」タブ）
2. **自分のコード**（下のセル）
3. **AIに投げた結果**（CSVを貼って「平均を出して」と頼む）

**ずれたら、どれかが間違っている。**どれが間違っていたかを突き止めるまでが課題である。
よくある原因は、①空欄の扱いが違う ②外れ値を除いたかどうか ③そもそも列を間違えている。

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# 自分のデータで、基本統計量を出す（mydf を読み込んでから実行）
# col = "ここに列名"
# print(f"件数     : {mydf[col].count()}")
# print(f"平均     : {mydf[col].mean():.2f}")
# print(f"中央値   : {mydf[col].median():.2f}")
# print(f"標準偏差 : {mydf[col].std():.2f}")

---
## 課題6（8点）

**このノートブック（コードと出力が残った状態）**を提出する。含めるもの：

- [ ] 自分のデータを読み込んだセルと、その出力
- [ ] 基本統計量（件数・平均・中央値・標準偏差）
- [ ] **元データのどこが汚かったかの記録**（1つ以上。なければ「なかった」と書いて理由を添える）
- [ ] 3通りの集計が一致したか。ずれた場合は、原因

**データファイルそのものは提出のみ。リポジトリには上げない。**（回答者の情報が含まれるため）

提出期限：次回授業の開始まで（遅れた場合は50%）